In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

### The goal of this project is to forecast the power consumption of the Eastern United States Region Service by PJM

"PJM Interconnection LLC (PJM) is a regional transmission organization (RTO) in the United States. It is part of the Eastern Interconnection grid operating an electric transmission system serving all or parts of Delaware, Illinois, Indiana, Kentucky, Maryland, Michigan, New Jersey, North Carolina, Ohio, Pennsylvania, Tennessee, Virginia, West Virginia, and the District of Columbia."

Source: Wikipedia (https://en.wikipedia.org/wiki/PJM_Interconnection)
Note: The dataset was not gotten from wikipedia. The only information gotten from wikipedia is the explanatio of PJM and the areas they service.

In [ ]:
# Store the power consumption dataset inside a pandas dataframe
df = pd.read_csv('PJME_hourly.csv')

# View the first few rows of the data
df.head()

In [ ]:
# Check the last few rows
df.tail()

In [ ]:
# Check for null data
df.isna().sum()

#### The output above shows that there is not missing data.

#### Let's get some further understanding of the data

In [ ]:
df.info()

In [ ]:
df.describe()

#### We have a total of 145366 rows of data

In [ ]:
# Let check for missing data using some seaborn visualization

sns.heatmap(df.isnull(), yticklabels=False, cbar= False, cmap= 'viridis')

#### The heat map also confirms that there is no missing data. If there was any missing data, we will see yellow dashes in its respective column.

In [ ]:
# Convert the 'Datetime' column to datetime format
df['Datetime'] = pd.to_datetime(df['Datetime'])

# Let's plot a graph that will show us the energy usage

fig, ax = plt.subplots(figsize=(10, 6))  # Set figure size
ax.plot(df['Datetime'], df['PJME_MW'], label='Power Consumption')
ax.set_xlabel('Date')
ax.set_ylabel('Power Consumption in MW')
ax.legend()
ax.set_title('Power Consumption in the Eastern USA From 2002 to 2017')

plt.show()

In [ ]:
sns.boxplot(data=df)

#### It looks like we have some spikes in the data. Let's break things down a bot further to see if we can decode them.

In [ ]:
# Let's start by creating new features

df['weekday_name'] = df['Datetime'].dt.day_name()
df['weekday'] = df['Datetime'].dt.weekday
df['month'] = df['Datetime'].dt.month
df['hour'] = df['Datetime'].dt.hour
df['year'] = df['Datetime'].dt.year

df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))  # Set figure size
ax.plot(df['weekday'], df['PJME_MW'], label='Power Consumption')
ax.set_xlabel('Day')
ax.set_ylabel('Power Consumption in MW')
ax.legend()
ax.set_title('Weekday Power Consumption in the Eastern USA From 2002 to 2017')

plt.show()

#### That plot is not very useful. Let's try a box plot

In [ ]:
fig, ax = plt.subplots(figsize=(20, 6))
sns.boxplot(data=df, x='weekday', y='PJME_MW')
ax.set_title('Weekday Energy Consumption')
plt.show()

#### We can see that Monday through Friday (0 - 4) have a similar power consumption, but the weekends, Saturday and Sunday (5 and 6), have a lower power consumption.

#### Let's see if we can get more insight using the hourly data.

In [ ]:
fig, ax = plt.subplots(figsize=(20, 6))
sns.boxplot(data=df, x='hour', y='PJME_MW')
ax.set_title('Hourly Energy Consumption')
plt.show()

#### Let's do the same thing for month

In [ ]:
fig, ax = plt.subplots(figsize=(20, 6))
sns.boxplot(data=df, x='month', y='PJME_MW')
ax.set_title('MW by hour')
plt.show()

#### It looks like the most power is used during the summer months

In [ ]:
# Let's see the pairplot of our new datframe to further understand any underlying relationships.

sns.pairplot(data=df)

In [ ]:
#Let's check the correlation
df.drop('weekday_name', axis=1).corr()

#### We are going to split the data into training and testing. We are not going to use sklearn to randamly split the data because we re trying to foecast. So, what we are going to do is split the data into old and new. the ols data will be the training data and it will be used to predict the test data ( the newer data).

In [ ]:
df.head()

In [ ]:
df_train = df.loc[df['Datetime'] < '2015-01-01']
df_test = df.loc[df['Datetime'] >= '2015-01-01']

Features = ['hour', 'weekday', 'month', 'year']
Target = 'PJME_MW'

X_train = df_train[Features]
y_train = df_train[Target]

X_test = df_test[Features]
y_test = df_test[Target]

#### We are going to use XGBoost for this. What is XGBoost?

XGBoost (eXtreme Gradient Boosting) is a powerful and efficient implementation of gradient boosting for supervised learning tasks, widely used in Python for both classification and regression problems. It is based on an ensemble of decision trees, where each new tree corrects the errors made by the previous ones. XGBoost uses advanced regularization (L1 and L2) to reduce overfitting and features a highly optimized algorithm that makes it both fast and scalable, suitable for large datasets. Its xgboost Python library offers easy integration with other machine learning libraries like scikit-learn, and supports functionalities like missing value handling, parallel processing, and model tuning via hyperparameters. Because of its strong predictive performance, XGBoost is popular in data science competitions and real-world applications.

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

In [ ]:
model = xgb.XGBRegressor(n_estimators=1000, early_stopping_rounds=50,
                      learning_rate=0.001)
model.fit(X_train, y_train,
        eval_set=[(X_train,  y_train), (X_test, y_test)],
        verbose=100)

In [ ]:
# Let us make predictions with our model
df_test['prediction'] = model.predict(X_test)

# Let us determine how good the model is
mse_train = mean_squared_error(y_train, model.predict(X_train))
mse_test = mean_squared_error(y_test, model.predict(X_test))

print(f"RMSE on the training data: {np.sqrt(mse_train)}\n")
print(f"RMSE on the test data: {np.sqrt(mse_test)}")

In [ ]:
# Let's us superimpose our predictions on the real data
df = df.merge(df_test['prediction'], how='left', left_index=True, right_index=True)

ax = df[['PJME_MW']].plot(figsize=(15, 5))
df['prediction'].plot(ax=ax, style='.')
plt.legend(['Truth Data', 'Predictions'])
ax.set_title('Raw Data and Prediction')
plt.show()

#### Considereing the fact that the root mean square on the training data and testing data are close. The model performed well, we may be able to make it better by a further preprocessing of more feature extraction.